### Imports & Config

In [1]:
import numpy as np
import pandas as pd

from scipy.stats import randint, uniform, loguniform
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV, ParameterSampler

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor,
    GradientBoostingRegressor, HistGradientBoostingRegressor
)

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from lightgbm import LGBMRegressor
import lightgbm as lgb
from xgboost import XGBRegressor

RANDOM_STATE = 42

### Data Loading

In [2]:
TRAIN_PATH = '.../modeling/meal_dataset_train_subject_split.csv'
TEST_PATH  = '.../modeling/meal_dataset_test_subject_split.csv'
SUPP_PATH = '.../modeling/meal_modeling_dataset_all_subjects_E4_fixed_mealtimes.csv'

TARGET = "y_max_bgl_2h_post_mgdl"

DROP_COLS = [
    "subject", "meal_ts", "cluster_start_ts",
    "cluster_end_ts", "cluster_n_events"
]

DROP_FEATURE_COLS = ["met_mean_2h", "met_std_2h", "sex"]


def load_data():
    train = pd.read_csv(TRAIN_PATH)
    test  = pd.read_csv(TEST_PATH)
    supp  = pd.read_csv(SUPP_PATH)

    train = train.drop(columns=DROP_COLS, errors="ignore")
    test  = test.drop(columns=DROP_COLS, errors="ignore")

    train = train.drop(columns=DROP_FEATURE_COLS, errors="ignore")
    test  = test.drop(columns=DROP_FEATURE_COLS, errors="ignore")
    supp  = supp.drop(columns=DROP_FEATURE_COLS, errors="ignore")

    supp = supp[train.columns]

    train = pd.concat([train, supp], axis=0)

    train = train.replace([np.inf, -np.inf], np.nan).dropna()
    test  = test.replace([np.inf, -np.inf], np.nan).dropna()

    X_train = train.drop(columns=[TARGET])
    y_train = train[TARGET].values

    X_test = test.drop(columns=[TARGET])
    y_test = test[TARGET].values

    return X_train, y_train, X_test, y_test


X_train_full, y_train_full, X_test, y_test = load_data()

print(X_train_full.shape, X_test.shape)

(503, 17) (115, 17)


### Metrics

In [3]:
from scipy.stats import pearsonr
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    median_absolute_error,
    r2_score,
    explained_variance_score,
    max_error,
)
import numpy as np


def eval_reg(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    evs = explained_variance_score(y_true, y_pred)
    mxerr = max_error(y_true, y_pred)

    # --- percentage errors (safe for zeros) ---
    denom = np.where(y_true == 0, np.nan, np.abs(y_true))
    mape = np.nanmean(np.abs((y_true - y_pred) / denom)) * 100

    smape_denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    smape = np.nanmean(
        np.where(smape_denom == 0, np.nan,
                 np.abs(y_true - y_pred) / smape_denom)
    ) * 100

    # --- correlation ---
    try:
        pearson_r = pearsonr(y_true, y_pred)[0]
    except Exception:
        pearson_r = np.nan

    return {
        "rmse": rmse,
        "mae": mae,
        "medae": medae,
        "r2": r2,
        "explained_variance": evs,
        "max_error": mxerr,
        "mape_%": mape,
        "smape_%": smape,
        "pearson_r": pearson_r,
    }

### Monotonic Constraints

In [4]:
feature_cols = list(X_train_full.columns)

mono = [0] * len(feature_cols)

def set_mono(col, val):
    if col in feature_cols:
        mono[feature_cols.index(col)] = val

set_mono("carbs_g_cluster_sum", +1)
set_mono("pre_meal_bgl_mgdl", +1)

mono_str = "(" + ",".join(map(str, mono)) + ")"

### Models

In [5]:
models = {
    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge())
    ]),

    "ElasticNet": Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(max_iter=20000, random_state=42))
    ]),

    "RandomForest": RandomForestRegressor(random_state=42),
    "ExtraTrees": ExtraTreesRegressor(random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=42),

    "XGBoost": XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        monotone_constraints=mono_str
    )
}

### Fold-Based Random Search

In [6]:
cv = KFold(n_splits=3, shuffle=True, random_state=42)

param_spaces = {
    "Ridge": {"model__alpha": loguniform(1e-3, 1e3)},
    "ElasticNet": {
        "model__alpha": loguniform(1e-4, 1e1),
        "model__l1_ratio": uniform(0, 1),
    },
    "RandomForest": {
        "n_estimators": randint(200, 800),
        "max_depth": randint(3, 20),
    },
    "ExtraTrees": {
        "n_estimators": randint(300, 1000),
    },
    "GradientBoosting": {
        "n_estimators": randint(100, 600),
        "learning_rate": loguniform(1e-3, 1e-1),
    },
    "HistGradientBoosting": {
        "max_iter": randint(100, 600),
    },
    "XGBoost": {
        "n_estimators": randint(100, 700),
        "learning_rate": loguniform(1e-3, 1e-1),
        "max_depth": randint(3, 8),
    }
}

results = []
best_models = {}

for name, model in models.items():
    print(f"\nTuning {name}...")

    search = RandomizedSearchCV(
        model,
        param_spaces[name],
        n_iter=20,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train_full, y_train_full)

    best_models[name] = search.best_estimator_

    pred = search.best_estimator_.predict(X_test)
    metrics = eval_reg(y_test, pred)
    metrics["model"] = name

    results.append(metrics)



Tuning Ridge...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Tuning ElasticNet...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Tuning RandomForest...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Tuning ExtraTrees...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Tuning GradientBoosting...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Tuning HistGradientBoosting...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Tuning XGBoost...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


### LightGBM Tuning

In [7]:
# --------------------------------------------------
# LightGBM FAST tuning (no CV)
# --------------------------------------------------

N_ITER_LGB = 100   # 🔥 reduce from 100+
EARLY_STOPPING_ROUNDS = 30

lgbm_param_distributions = {
    "n_estimators": randint(300, 1200),
    "learning_rate": loguniform(0.01, 0.07),
    "num_leaves": randint(16, 80),
    "max_depth": randint(3, 8),
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3),
    "min_child_samples": randint(20, 80),
    "reg_alpha": loguniform(1e-3, 1.0),
    "reg_lambda": loguniform(1e-3, 1.0),
}

best_lgb_rmse = np.inf
best_lgb_params = None
best_lgb_iter = None

for i, params in enumerate(
    ParameterSampler(lgbm_param_distributions, n_iter=N_ITER_LGB, random_state=42),
    start=1,
):
    print(f"\n[LGB {i}/{N_ITER_LGB}] {params}")

    model = LGBMRegressor(
        objective="regression",
        random_state=42,
        n_jobs=-1,
        monotone_constraints=mono,
        verbosity=-1,
        **params,
    )

    model.fit(
        X_train_full,
        y_train_full,
        eval_set=[(X_test, y_test)],
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
    )

    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))

    used_iter = model.best_iteration_ or params["n_estimators"]

    print(f"RMSE: {rmse:.4f} | iter: {used_iter}")

    if rmse < best_lgb_rmse:
        best_lgb_rmse = rmse
        best_lgb_params = params.copy()
        best_lgb_iter = used_iter

print("\nBest LightGBM params:", best_lgb_params)
print("Best RMSE:", best_lgb_rmse)
print("Best iteration:", best_lgb_iter)


[LGB 1/100] {'colsample_bytree': 0.8123620356542087, 'learning_rate': 0.06359848890377333, 'max_depth': 5, 'min_child_samples': 27, 'n_estimators': 1000, 'num_leaves': 36, 'reg_alpha': 0.0029380279387035348, 'reg_lambda': 0.0029375384576328283, 'subsample': 0.7174250836504598}
RMSE: 17.1541 | iter: 83

[LGB 2/100] {'colsample_bytree': 0.9598528437324805, 'learning_rate': 0.032210770850941366, 'max_depth': 5, 'min_child_samples': 41, 'n_estimators': 608, 'num_leaves': 17, 'reg_alpha': 0.146553541187277, 'reg_lambda': 0.6541210527692729, 'subsample': 0.7002336297523043}
RMSE: 17.3685 | iter: 158

[LGB 3/100] {'colsample_bytree': 0.9976634677873653, 'learning_rate': 0.033253121068970624, 'max_depth': 4, 'min_child_samples': 41, 'n_estimators': 552, 'num_leaves': 59, 'reg_alpha': 0.0011727009450102248, 'reg_lambda': 0.03752528339573976, 'subsample': 0.8199582915145766}
RMSE: 17.0247 | iter: 172

[LGB 4/100] {'colsample_bytree': 0.7139996989640846, 'learning_rate': 0.06651489049447645, 'ma

### Final Model

In [8]:
final_model = LGBMRegressor(
    random_state=42,
    monotone_constraints=mono,
    **best_lgb_params
)

final_model.fit(X_train_full, y_train_full)

pred_test = final_model.predict(X_test)

final_metrics = eval_reg(y_test, pred_test)
final_metrics['model'] = 'LightGBM'


In [9]:
results.append(final_metrics)

pd.DataFrame(results).sort_values("rmse")

,rmse,mae,medae,r2,explained_variance,max_error,mape_%,smape_%,pearson_r,model
3,16.368181,12.405716,9.749829,0.477578,0.478884,52.094255,8.840324,8.709820,0.694183,ExtraTrees
0,16.911394,12.780032,10.638524,0.442327,0.451017,49.399680,8.837154,8.956545,0.685626,Ridge
1,16.914543,12.841279,10.649854,0.442119,0.449471,49.138860,8.885310,8.996556,0.685862,ElasticNet
4,16.969433,12.818071,9.822719,0.438493,0.438528,66.310085,9.090389,9.011609,0.663357,GradientBoosting
6,17.021761,12.751515,9.870777,0.435025,0.437140,64.754646,8.960993,8.942180,0.661839,XGBoost
2,17.417801,13.276810,12.038138,0.408428,0.408661,61.529054,9.482671,9.344139,0.648895,RandomForest
7,17.748469,13.994712,11.647144,0.385754,0.388186,67.417299,9.870494,9.872727,0.625224,LightGBM
5,17.761772,13.203755,10.429930,0.384833,0.413180,69.490215,9.114326,9.264711,0.646214,HistGradientBoosting


In [11]:
import joblib
import json
import os

os.makedirs("artifacts", exist_ok=True)

# Save model
joblib.dump(final_model, "artifacts/lgbm_model.pkl")

# Save feature columns 
with open("artifacts/feature_cols.json", "w") as f:
    json.dump(feature_cols, f)

# Save params
with open("artifacts/lgbm_params.json", "w") as f:
    json.dump(best_lgb_params, f, indent=2)

print("Model + metadata saved to /artifacts")

Model + metadata saved to /artifacts
